In [ ]:
# Cell 1: Imports and constants
import numpy as np
import pandas as pd
from datetime import date, timedelta
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# DATA_PATH for chart xlsx — update to actual file path
DATA_PATH = "sofr_history.xlsx"

# Tenors in months (18M=18, 2Y=24)
TENORS = [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 18, 24]
TENOR_LABELS = ["1M", "2M", "3M", "4M", "5M", "6M", "9M", "10M", "11M", "12M", "18M", "2Y"]

# EFFR (starting rate, TNA, decimal)
EFFR_DEFAULT = 4.33 / 100

# OIS mid defaults ~2 cuts by Dec26 (TNA, decimal)
# Approximate market implied levels for each tenor
OIS_DEFAULTS = {
    1:  4.33,
    2:  4.30,
    3:  4.25,
    4:  4.20,
    5:  4.15,
    6:  4.10,
    9:  4.00,
    10: 3.97,
    11: 3.94,
    12: 3.91,
    18: 3.80,
    24: 3.72,
}

# FOMC decision dates (effective next biz day)
FOMC_DATES = [
    date(2026, 1, 28),
    date(2026, 3, 18),
    date(2026, 4, 29),
    date(2026, 6, 17),
    date(2026, 7, 29),
    date(2026, 9, 16),
    date(2026, 10, 28),
    date(2026, 12, 9),
    date(2027, 1, 27),
    date(2027, 3, 17),
    date(2027, 4, 28),
    date(2027, 6, 9),
    date(2027, 7, 28),
]
N_MEETINGS = len(FOMC_DATES)

print(f"SOFR OIS FV Model loaded. {N_MEETINGS} FOMC meetings. Tenors: {TENOR_LABELS}")


In [ ]:
# Cell 2: Holiday and date utilities

def us_holidays(year):
    """Return set of US SIFMA holidays for given year."""
    h = set()
    # Jan 1
    h.add(date(year, 1, 1))
    # MLK: 3rd Monday Jan
    h.add(_nth_weekday(year, 1, 0, 3))
    # Presidents: 3rd Monday Feb
    h.add(_nth_weekday(year, 2, 0, 3))
    # Good Friday
    h.add(_good_friday(year))
    # Memorial: last Monday May
    h.add(_last_weekday(year, 5, 0))
    # Juneteenth
    h.add(date(year, 6, 19))
    # Jul 4
    h.add(date(year, 7, 4))
    # Labor: 1st Monday Sep
    h.add(_nth_weekday(year, 9, 0, 1))
    # Columbus: 2nd Monday Oct
    h.add(_nth_weekday(year, 10, 0, 2))
    # Veterans Day
    h.add(date(year, 11, 11))
    # Thanksgiving: 4th Thursday Nov
    h.add(_nth_weekday(year, 11, 3, 4))
    # Christmas
    h.add(date(year, 12, 25))
    return h

def _nth_weekday(year, month, weekday, n):
    """n-th occurrence of weekday (0=Mon) in month."""
    d = date(year, month, 1)
    count = 0
    while True:
        if d.weekday() == weekday:
            count += 1
            if count == n:
                return d
        d += timedelta(1)

def _last_weekday(year, month, weekday):
    """Last occurrence of weekday in month."""
    if month == 12:
        last = date(year + 1, 1, 1) - timedelta(1)
    else:
        last = date(year, month + 1, 1) - timedelta(1)
    while last.weekday() != weekday:
        last -= timedelta(1)
    return last

def _good_friday(year):
    """Good Friday via anonymous Gregorian algorithm."""
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    easter = date(year, month, day)
    return easter - timedelta(2)

# Build holiday set for 2025-2028
_ALL_HOLIDAYS = set()
for _yr in range(2025, 2029):
    _ALL_HOLIDAYS |= us_holidays(_yr)

def is_bizday(d):
    return d.weekday() < 5 and d not in _ALL_HOLIDAYS

def next_bizday(d):
    """Next business day after d."""
    d = d + timedelta(1)
    while not is_bizday(d):
        d += timedelta(1)
    return d

def add_bizdays(d, n):
    """Add n business days to d."""
    while n > 0:
        d = next_bizday(d)
        n -= 1
    return d

def modfol(d):
    """Modified following: next biz day, but stay in month."""
    orig_month = d.month
    nd = d
    while not is_bizday(nd):
        nd += timedelta(1)
    if nd.month != orig_month:
        nd = d
        while not is_bizday(nd):
            nd -= timedelta(1)
    return nd

def add_months(d, m):
    """Add m months to date, return result."""
    month = d.month - 1 + m
    year = d.year + month // 12
    month = month % 12 + 1
    import calendar
    day = min(d.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)

def spot_date(val_date):
    """T+2 US biz days."""
    return add_bizdays(val_date, 2)

def tenor_end(spot, months):
    """ModFol end date for tenor in months from spot."""
    raw = add_months(spot, months)
    return modfol(raw)

# FOMC effective dates: next biz day after decision
FOMC_EFFECTIVE = [next_bizday(d) for d in FOMC_DATES]

print("Date utilities ready.")
print("FOMC effective dates:")
for dec, eff in zip(FOMC_DATES, FOMC_EFFECTIVE):
    print(f"  Decision {dec} -> Effective {eff}")


In [ ]:
# Cell 3: OIS pricing engine

def compute_fv_for_path(val_date, path_bps, effr, tenor_months):
    """
    Compute FV (TNA) for a single path and single tenor.
    path_bps: list of 13 bp changes at each FOMC meeting (cumulative from effr).
    Returns FV as decimal (e.g. 0.0433).
    """
    spot = spot_date(val_date)
    end = tenor_end(spot, tenor_months)
    D = (end - spot).days

    # Build rate schedule: list of (start_date, end_date, rate)
    # Rate changes on FOMC_EFFECTIVE dates
    rate_schedule = _build_rate_schedule(spot, end, effr, path_bps)

    # Compound: product of (1 + r * n / 360) per block
    accrual = 1.0
    for (block_start, block_end, rate) in rate_schedule:
        n = (block_end - block_start).days
        if n > 0:
            accrual *= (1.0 + rate * n / 360.0)

    # For <=12M: zero coupon FV = (accrual - 1) * 360 / D
    # For 18M: annual pay, two periods
    # For 2Y: annual pay, two periods
    if tenor_months <= 12:
        fv = (accrual - 1.0) * 360.0 / D
    elif tenor_months == 18:
        fv = _annual_pay_fv(spot, end, effr, path_bps, periods=2, total_months=18)
    else:
        # 2Y
        fv = _annual_pay_fv(spot, end, effr, path_bps, periods=2, total_months=24)
    return fv

def _build_rate_schedule(start, end, effr, path_bps):
    """
    Build list of (block_start, block_end, rate) from start to end.
    Rate starts at effr, changes by path_bps[i] bps at FOMC_EFFECTIVE[i].
    path_bps is cumulative change in bps (each element = bp move at that meeting).
    """
    # Cumulative rate levels at each FOMC
    cum_bps = 0.0
    rate_changes = []  # (effective_date, new_rate)
    for i, eff_date in enumerate(FOMC_EFFECTIVE):
        cum_bps += path_bps[i]
        new_rate = effr + cum_bps / 10000.0
        rate_changes.append((eff_date, new_rate))

    # Filter to those within [start, end]
    blocks = []
    current_rate = effr
    current_start = start

    for eff_date, new_rate in rate_changes:
        if eff_date <= start:
            # Change already happened before our window
            current_rate = new_rate
            continue
        if eff_date >= end:
            break
        # Close current block at eff_date
        blocks.append((current_start, eff_date, current_rate))
        current_start = eff_date
        current_rate = new_rate

    # Final block to end
    blocks.append((current_start, end, current_rate))
    return blocks

def _annual_pay_fv(spot, end, effr, path_bps, periods, total_months):
    """
    Annual pay FV for 18M (2 periods: 0-12M, 12-18M) and 2Y (0-12M, 12-24M).
    FV = par rate such that PV of fixed = PV of float.
    Float accrual per period, discount = product of all blocks to coupon date.
    """
    # Period end dates
    period_months = total_months // periods
    period_ends = []
    for p in range(1, periods + 1):
        period_ends.append(tenor_end(spot, period_months * p))

    # Float accrual factor per period
    float_accr = []
    prev = spot
    for pe in period_ends:
        sched = _build_rate_schedule(prev, pe, effr, path_bps)
        acc = 1.0
        for (bs, be, r) in sched:
            n = (be - bs).days
            if n > 0:
                acc *= (1.0 + r * n / 360.0)
        float_accr.append(acc)
        prev = pe

    # Discount factors: product from spot to each period end
    # DF(0,T_k) = 1 / product of all blocks spot->T_k
    cumulative = 1.0
    dfs = []
    prev = spot
    for pe in period_ends:
        sched = _build_rate_schedule(prev, pe, effr, path_bps)
        for (bs, be, r) in sched:
            n = (be - bs).days
            if n > 0:
                cumulative *= (1.0 + r * n / 360.0)
        dfs.append(1.0 / cumulative)
        prev = pe

    # Float PV = sum over periods: (float_accr_period - 1) * DF(0, period_end)
    # where float_accr_period is the period compounding (not cumulative)
    # Recalculate per-period
    cum_acc = 1.0
    prev = spot
    period_accs = []
    for pe in period_ends:
        sched = _build_rate_schedule(prev, pe, effr, path_bps)
        acc = 1.0
        for (bs, be, r) in sched:
            n = (be - bs).days
            if n > 0:
                acc *= (1.0 + r * n / 360.0)
        period_accs.append(acc)
        prev = pe

    # Par swap rate: C * sum(alpha_k * DF_k) = sum((acc_k - 1) * DF_k)
    # alpha_k = (period_end_k - period_end_{k-1}).days / 360
    period_starts = [spot] + period_ends[:-1]
    alphas = [(period_ends[i] - period_starts[i]).days / 360.0 for i in range(periods)]

    float_pv = sum((period_accs[i] - 1.0) * dfs[i] for i in range(periods))
    annuity = sum(alphas[i] * dfs[i] for i in range(periods))
    fv = float_pv / annuity
    return fv

def compute_all_fvs(val_date, paths_df, effr):
    """
    Compute FV for all paths × all tenors.
    paths_df: DataFrame with columns = FOMC meeting labels, rows = paths, values = bp changes.
    Returns DataFrame (paths × tenors) in decimal.
    """
    result = {}
    for path_name, row in paths_df.iterrows():
        path_bps = list(row.values)
        fvs = []
        for tm in TENORS:
            fv = compute_fv_for_path(val_date, path_bps, effr, tm)
            fvs.append(fv)
        result[path_name] = fvs
    df = pd.DataFrame(result, index=TENOR_LABELS).T
    return df

print("Pricing engine ready.")


In [ ]:
# Cell 4: WIRP binary tree bootstrap

def wirp_bootstrap(val_date, ois_mids_pct, effr, increment_bps=25):
    """
    Sequential bootstrap: solve p_k (prob of cut) meeting by meeting.
    Uses shortest OIS tenor that straddles each meeting.
    ois_mids_pct: dict {tenor_months: rate_pct}.
    Returns DataFrame with columns: Meeting, P(Cut)%, P(Hold)%, P(Hike)%, ImpliedRate, CumChg(bps).
    """
    spot = spot_date(val_date)
    inc = increment_bps / 10000.0

    # Build sorted tenor end dates
    tenor_ends = {tm: tenor_end(spot, tm) for tm in TENORS}

    # For each meeting, find shortest tenor end >= effective date
    results = []
    cum_rate = effr  # running implied rate
    cum_chg_bps = 0.0

    # We'll accumulate rates meeting by meeting using solved p_k
    # rates[i] = rate in effect between meeting i effective and meeting i+1 effective
    solved_rates = []  # rate applying from meeting i effective date

    for k, (dec_date, eff_date) in enumerate(zip(FOMC_DATES, FOMC_EFFECTIVE)):
        # Find shortest tenor that straddles: tenor_end >= eff_date
        straddling = [(tm, te) for tm, te in tenor_ends.items() if te >= eff_date]
        if not straddling:
            # Beyond all tenors — use last known rate
            p_cut = 0.0
            results.append({
                "Meeting": dec_date,
                "P(Cut)%": round(p_cut * 100, 1),
                "P(Hold)%": round((1 - p_cut) * 100, 1),
                "P(Hike)%": 0.0,
                "ImpliedRate(%)": round(cum_rate * 100, 4),
                "CumChg(bps)": round(cum_chg_bps, 1),
            })
            solved_rates.append(cum_rate)
            continue

        straddling.sort(key=lambda x: x[0])
        use_tenor, use_end = straddling[0]
        ois_rate = ois_mids_pct[use_tenor] / 100.0

        # Build implied accrual from ois_rate
        D = (use_end - spot).days
        if use_tenor <= 12:
            target_acc = 1.0 + ois_rate * D / 360.0
        else:
            # For annual pay, approximate: use zero-coupon equivalent
            target_acc = (1.0 + ois_rate * D / 360.0)

        # Accrual from spot to eff_date using solved_rates
        acc_to_eff = _accrual_with_solved(spot, eff_date, effr, solved_rates)

        # After eff_date: two branches with p_cut probability
        # cut branch: rate = cum_rate - inc
        # hold branch: rate = cum_rate
        # Solve p from: acc_to_eff * [p*(1+r_cut*n/360) + (1-p)*(1+r_hold*n/360)] * remainder = target_acc
        # Simplify: solve for p in the meeting's block only
        # Days from eff_date to next eff_date (or tenor end if last)
        if k + 1 < N_MEETINGS:
            next_eff = FOMC_EFFECTIVE[k + 1]
        else:
            next_eff = use_end

        block_end = min(next_eff, use_end)
        n_block = (block_end - eff_date).days

        # Remainder after block (rate = cum_rate for both branches, same)
        # Actually after next meeting, rate converges — for bootstrap use this simplified approach
        r_hold = cum_rate
        r_cut = cum_rate - inc

        if n_block > 0 and use_end > block_end:
            # There's remainder from block_end to use_end at current rate (pre-next-meeting rate)
            n_rem = (use_end - block_end).days
            rem_acc_hold = (1.0 + r_hold * n_rem / 360.0)
            rem_acc_cut = (1.0 + r_cut * n_rem / 360.0)

            hold_total = acc_to_eff * (1.0 + r_hold * n_block / 360.0) * rem_acc_hold
            cut_total = acc_to_eff * (1.0 + r_cut * n_block / 360.0) * rem_acc_cut

            if abs(hold_total - cut_total) > 1e-12:
                p_cut = (target_acc - hold_total) / (cut_total - hold_total)
            else:
                p_cut = 0.5
        elif n_block > 0:
            hold_total = acc_to_eff * (1.0 + r_hold * n_block / 360.0)
            cut_total = acc_to_eff * (1.0 + r_cut * n_block / 360.0)
            if abs(hold_total - cut_total) > 1e-12:
                p_cut = (target_acc - hold_total) / (cut_total - hold_total)
            else:
                p_cut = 0.5
        else:
            p_cut = 0.0

        # Clamp to [0, 1]
        p_cut = max(0.0, min(1.0, p_cut))

        # Expected rate after this meeting
        exp_rate = p_cut * r_cut + (1 - p_cut) * r_hold
        cum_chg_bps += p_cut * (-increment_bps)
        cum_rate = exp_rate
        solved_rates.append(exp_rate)

        results.append({
            "Meeting": dec_date,
            "P(Cut)%": round(p_cut * 100, 1),
            "P(Hold)%": round((1 - p_cut) * 100, 1),
            "P(Hike)%": 0.0,
            "ImpliedRate(%)": round(cum_rate * 100, 4),
            "CumChg(bps)": round(cum_chg_bps, 1),
        })

    return pd.DataFrame(results)

def _accrual_with_solved(spot, end, effr, solved_rates):
    """Compute accrual from spot to end using solved expected rates per meeting block."""
    acc = 1.0
    current_rate = effr
    current_start = spot
    for i, eff_date in enumerate(FOMC_EFFECTIVE):
        if eff_date <= spot:
            if i < len(solved_rates):
                current_rate = solved_rates[i]
            continue
        if eff_date >= end:
            break
        n = (eff_date - current_start).days
        if n > 0:
            acc *= (1.0 + current_rate * n / 360.0)
        current_start = eff_date
        if i < len(solved_rates):
            current_rate = solved_rates[i]
    n = (end - current_start).days
    if n > 0:
        acc *= (1.0 + current_rate * n / 360.0)
    return acc

print("WIRP bootstrap ready.")


In [ ]:
# Cell 5: Default paths and state management

# FOMC meeting labels for path table columns
MTG_LABELS = [d.strftime("%b-%y") for d in FOMC_DATES]

# Default paths: name -> list of 13 bp changes (one per FOMC meeting)
# Negative = cut, positive = hike
def _zeros():
    return [0] * N_MEETINGS

DEFAULT_PATHS = {
    "Hold":              _zeros(),
    "Jun-26":            [0, 0, 0, -25,  0,   0,  0,  0,  0,  0,  0,  0,  0],
    "Jul-26":            [0, 0, 0,  0, -25,   0,  0,  0,  0,  0,  0,  0,  0],
    "Sep-26":            [0, 0, 0,  0,  0, -25,   0,  0,  0,  0,  0,  0,  0],
    "Oct-26":            [0, 0, 0,  0,  0,   0, -25,  0,  0,  0,  0,  0,  0],
    "Dec-26":            [0, 0, 0,  0,  0,   0,  0, -25,  0,  0,  0,  0,  0],
    "Jun/Sep-26":        [0, 0, 0, -25,  0, -25,   0,  0,  0,  0,  0,  0,  0],
    "Jun/Oct-26":        [0, 0, 0, -25,  0,   0, -25,  0,  0,  0,  0,  0,  0],
    "Jul/Sep-26":        [0, 0, 0,  0, -25, -25,   0,  0,  0,  0,  0,  0,  0],
    "Jun/Sep/Dec-26":    [0, 0, 0, -25,  0, -25,   0, -25,  0,  0,  0,  0,  0],
    "Jun-50":            [0, 0, 0, -50,  0,   0,   0,  0,  0,  0,  0,  0,  0],
    "Apr+25":            [0, 0, 25,  0,  0,   0,   0,  0,  0,  0,  0,  0,  0],
    "Jun/Sep/Dec/Mar27": [0, 0, 0, -25,  0, -25,   0, -25, -25,  0,  0,  0,  0],
}

# Global mutable state — paths as DataFrame
_paths_df = pd.DataFrame(DEFAULT_PATHS, index=MTG_LABELS).T.copy()

# OIS mids state (pct)
_ois_mids = dict(OIS_DEFAULTS)

# EFFR state
_effr = EFFR_DEFAULT

# Weights state (equal by default, user editable)
_weights = {name: 100.0 / len(DEFAULT_PATHS) for name in DEFAULT_PATHS}

# Val date
_val_date = date.today()

print(f"Default paths loaded: {list(_paths_df.index)}")
print(f"Val date: {_val_date}")


In [ ]:
# Cell 6: FV Table + Deltas rendering

def render_fv_table(fv_df, mkt_rates_pct, weights):
    """
    Render HTML table: paths x tenors, FV in bps (vs mkt), color |delta|>4bp.
    fv_df: DataFrame (paths x tenors), values in decimal.
    mkt_rates_pct: dict {tenor_months: rate_pct}.
    weights: dict {path_name: weight_pct}.
    Returns HTML string.
    """
    mkt = np.array([mkt_rates_pct[tm] / 100.0 for tm in TENORS])
    fv_arr = fv_df.values  # shape (n_paths, n_tenors)
    path_names = list(fv_df.index)
    n_paths = len(path_names)

    # Weighted FV
    wt_arr = np.array([weights.get(p, 0.0) for p in path_names])
    wt_total = wt_arr.sum()
    if wt_total > 0:
        wt_norm = wt_arr / wt_total
    else:
        wt_norm = np.ones(n_paths) / n_paths
    weighted_fv = wt_norm @ fv_arr  # shape (n_tenors,)

    # Delta = (Mkt - FV) * 10000 bps
    delta_arr = (mkt[np.newaxis, :] - fv_arr) * 10000.0

    html = ["<style>",
            "table.fvtable {border-collapse:collapse; font-size:12px; font-family:monospace;}",
            "table.fvtable th, table.fvtable td {border:1px solid #555; padding:3px 6px; text-align:right;}",
            "table.fvtable th {background:#2a2a3e; color:#ccc;}",
            "tr.mkt-row td {background:#1a3a5c !important; color:#7ab7ff; font-weight:bold;}",
            "tr.fv-row td {background:#3a3510 !important; color:#ffe066; font-weight:bold;}",
            ".green-cell {background-color:#1a4a1a; color:#66ff66;}",
            ".red-cell {background-color:#4a1a1a; color:#ff6666;}",
            "</style>"]

    html.append('<table class="fvtable">')
    # Header
    html.append("<tr><th>Path</th><th>Wt%</th>")
    for tl in TENOR_LABELS:
        html.append(f"<th>{tl}</th>")
    html.append("</tr>")

    # Path rows
    for i, pname in enumerate(path_names):
        html.append("<tr>")
        html.append(f"<td style='text-align:left'>{pname}</td>")
        html.append(f"<td>{weights.get(pname, 0.0):.1f}</td>")
        for j in range(len(TENORS)):
            fv_pct = fv_arr[i, j] * 100.0
            delta = delta_arr[i, j]
            cell_class = ""
            if delta > 4.0:
                cell_class = "green-cell"
            elif delta < -4.0:
                cell_class = "red-cell"
            html.append(f'<td class="{cell_class}">{fv_pct:.3f}</td>')
        html.append("</tr>")

    # Mkt row
    html.append('<tr class="mkt-row"><td>Mkt</td><td></td>')
    for tm in TENORS:
        html.append(f"<td>{mkt_rates_pct[tm]:.3f}</td>")
    html.append("</tr>")

    # FV (weighted) row
    html.append('<tr class="fv-row"><td>FV (wtd)</td><td></td>')
    for j in range(len(TENORS)):
        html.append(f"<td>{weighted_fv[j]*100:.3f}</td>")
    html.append("</tr>")

    # Mkt-FV bps row
    html.append("<tr><td><b>Mkt-FV(bps)</b></td><td></td>")
    for j in range(len(TENORS)):
        diff = (mkt[j] - weighted_fv[j]) * 10000.0
        color = "#66ff66" if diff > 0 else "#ff6666"
        html.append(f'<td style="color:{color};font-weight:bold">{diff:+.1f}</td>')
    html.append("</tr>")

    # High delta row
    html.append("<tr><td><b>HighDelta(bps)</b></td><td></td>")
    for j in range(len(TENORS)):
        hd = delta_arr[:, j].max()
        html.append(f"<td>{hd:+.1f}</td>")
    html.append("</tr>")

    # Low delta row
    html.append("<tr><td><b>LowDelta(bps)</b></td><td></td>")
    for j in range(len(TENORS)):
        ld = delta_arr[:, j].min()
        html.append(f"<td>{ld:+.1f}</td>")
    html.append("</tr>")

    html.append("</table>")
    return "".join(html)

print("FV table renderer ready.")


In [ ]:
# Cell 7: Spread RV and Butterfly RV renderers

# Key spread pairs: (near_tenor_months, far_tenor_months, label)
SPREAD_PAIRS = [
    (1, 3, "1x3"),
    (2, 6, "2x6"),
    (3, 9, "3x9"),
    (6, 12, "6x12"),
    (9, 18, "9x18M"),
    (12, 24, "12Mx2Y"),
    (1, 6, "1x6"),
    (1, 12, "1x12"),
    (3, 12, "3x12"),
    (6, 18, "6x18M"),
    (6, 24, "6x2Y"),
    (12, 18, "12Mx18M"),
]

# Butterfly definitions: (wing1, body, wing2, label)
FLY_TRIPLETS = [
    (1, 3, 6, "1/3/6"),
    (3, 6, 9, "3/6/9"),
    (6, 9, 12, "6/9/12"),
    (3, 6, 12, "3/6/12"),
    (6, 12, 18, "6/12/18"),
    (9, 12, 24, "9/12/2Y"),
    (12, 18, 24, "12/18/2Y"),
]

def _tenor_idx(months):
    return TENORS.index(months)

def _spread_bps(rates_row, t1, t2):
    """Spread = far - near in bps."""
    return (rates_row[_tenor_idx(t2)] - rates_row[_tenor_idx(t1)]) * 10000.0

def _fly_bps(rates_row, t1, t2, t3):
    """Butterfly = near + far - 2*body in bps."""
    return (rates_row[_tenor_idx(t1)] + rates_row[_tenor_idx(t3)] - 2 * rates_row[_tenor_idx(t2)]) * 10000.0

def render_spread_rv(fv_df, mkt_rates_pct, weights, selected_view="FV"):
    """
    Render Spread RV table.
    selected_view: path name or 'FV' (weighted).
    Cell: path_spd (pickup) / mkt_spd bps. Bold if |pickup|>4bp.
    """
    mkt = np.array([mkt_rates_pct[tm] / 100.0 for tm in TENORS])
    path_names = list(fv_df.index)

    # Compute weighted FV
    wt_arr = np.array([weights.get(p, 0.0) for p in path_names])
    wt_total = wt_arr.sum()
    wt_norm = wt_arr / wt_total if wt_total > 0 else np.ones(len(path_names)) / len(path_names)
    weighted_fv = wt_norm @ fv_df.values

    # Select row to display
    if selected_view == "FV":
        display_rates = weighted_fv
        display_name = "FV (wtd)"
    else:
        if selected_view in fv_df.index:
            display_rates = fv_df.loc[selected_view].values
        else:
            display_rates = weighted_fv
        display_name = selected_view

    html = ["<style>",
            "table.rvtable {border-collapse:collapse; font-size:12px; font-family:monospace;}",
            "table.rvtable th, table.rvtable td {border:1px solid #555; padding:3px 8px; text-align:center;}",
            "table.rvtable th {background:#2a2a3e; color:#ccc;}",
            ".bold-pickup {font-weight:bold;}",
            "</style>",
            '<table class="rvtable">',
            f"<tr><th>Spread</th><th>{display_name} (bps)</th><th>Mkt (bps)</th><th>Pickup (bps)</th></tr>"]

    for t1, t2, label in SPREAD_PAIRS:
        path_spd = _spread_bps(display_rates, t1, t2)
        mkt_spd = _spread_bps(mkt, t1, t2)
        pickup = path_spd - mkt_spd
        bold = "bold-pickup" if abs(pickup) > 4.0 else ""
        html.append(f'<tr class="{bold}"><td>{label}</td><td>{path_spd:.1f}</td><td>{mkt_spd:.1f}</td><td>{pickup:+.1f}</td></tr>')

    html.append("</table>")
    return "".join(html)

def render_fly_rv(fv_df, mkt_rates_pct, weights, selected_view="FV"):
    """Render Butterfly RV table."""
    mkt = np.array([mkt_rates_pct[tm] / 100.0 for tm in TENORS])
    path_names = list(fv_df.index)

    wt_arr = np.array([weights.get(p, 0.0) for p in path_names])
    wt_total = wt_arr.sum()
    wt_norm = wt_arr / wt_total if wt_total > 0 else np.ones(len(path_names)) / len(path_names)
    weighted_fv = wt_norm @ fv_df.values

    if selected_view == "FV":
        display_rates = weighted_fv
        display_name = "FV (wtd)"
    else:
        display_rates = fv_df.loc[selected_view].values if selected_view in fv_df.index else weighted_fv
        display_name = selected_view

    html = ["<style>",
            "table.flytable {border-collapse:collapse; font-size:12px; font-family:monospace;}",
            "table.flytable th, table.flytable td {border:1px solid #555; padding:3px 8px; text-align:center;}",
            "table.flytable th {background:#2a2a3e; color:#ccc;}",
            "</style>",
            '<table class="flytable">',
            f"<tr><th>Fly</th><th>{display_name} (bps)</th><th>Mkt (bps)</th><th>Pickup (bps)</th></tr>"]

    for t1, t2, t3, label in FLY_TRIPLETS:
        path_fly = _fly_bps(display_rates, t1, t2, t3)
        mkt_fly = _fly_bps(mkt, t1, t2, t3)
        pickup = path_fly - mkt_fly
        bold = "font-weight:bold;" if abs(pickup) > 4.0 else ""
        html.append(f'<tr><td>{label}</td><td style="{bold}">{path_fly:.1f}</td><td>{mkt_fly:.1f}</td><td style="{bold}">{pickup:+.1f}</td></tr>')

    html.append("</table>")
    return "".join(html)

print("Spread/Fly RV renderers ready.")


In [ ]:
# Cell 8: DV01 and Hedge calculator

def compute_dv01(val_date, tenor_months, notional, rate_pct):
    """
    DV01 = N * 0.0001 * (D/360) / (1 + K * D/360).
    D = calendar days spot to maturity.
    K = rate (decimal).
    Sign: recv = negative, pay = positive.
    """
    spot = spot_date(val_date)
    end = tenor_end(spot, tenor_months)
    D = (end - spot).days
    K = rate_pct / 100.0
    dv01 = notional * 0.0001 * (D / 360.0) / (1.0 + K * D / 360.0)
    return dv01

def make_dv01_tab(val_date, mkt_rates_pct):
    """Build DV01 display widget."""
    notional_w = widgets.FloatText(value=10_000_000, description="Notional USD:", style={"description_width": "120px"})
    sign_w = widgets.ToggleButtons(options=["Receiver (-)","Payer (+)"], description="Direction:", style={"description_width": "80px"})
    out_dv01 = widgets.Output()

    def update_dv01(_=None):
        n = notional_w.value
        sign = -1 if sign_w.index == 0 else 1
        rows = []
        for tm, tl in zip(TENORS, TENOR_LABELS):
            rate = mkt_rates_pct[tm]
            dv01 = sign * compute_dv01(val_date, tm, n, rate)
            rows.append({"Tenor": tl, "Rate(%)": f"{rate:.3f}", "DV01($)": f"{dv01:,.0f}"})
        df = pd.DataFrame(rows).set_index("Tenor")
        with out_dv01:
            out_dv01.clear_output(wait=True)
            display(df)

    notional_w.observe(update_dv01, "value")
    sign_w.observe(update_dv01, "value")
    update_dv01()

    # Hedge solver section
    hedge_title = widgets.HTML("<b>Hedge Solver</b>")
    mode_w = widgets.RadioButtons(options=["Spread","Fly"], description="Mode:", style={"description_width": "60px"})
    leg1_w = widgets.Dropdown(options=TENOR_LABELS, description="Leg1:", value="3M")
    leg2_w = widgets.Dropdown(options=TENOR_LABELS, description="Leg2:", value="6M")
    leg3_w = widgets.Dropdown(options=TENOR_LABELS, description="Leg3:", value="9M")
    anchor_w = widgets.RadioButtons(options=["Leg1","Leg2","Leg3"], description="Anchor:")
    anchor_n_w = widgets.FloatText(value=10_000_000, description="Anchor N:", style={"description_width": "80px"})
    target_dv01_w = widgets.FloatText(value=0.0, description="Target DV01:", style={"description_width": "100px"})
    solve_btn = widgets.Button(description="Solve Hedge", button_style="primary")
    out_hedge = widgets.Output()

    def solve_hedge(_):
        with out_hedge:
            out_hedge.clear_output(wait=True)
            try:
                is_fly = mode_w.value == "Fly"
                t1 = TENORS[TENOR_LABELS.index(leg1_w.value)]
                t2 = TENORS[TENOR_LABELS.index(leg2_w.value)]
                t3 = TENORS[TENOR_LABELS.index(leg3_w.value)] if is_fly else None

                dv01_1 = compute_dv01(val_date, t1, 1e6, mkt_rates_pct[t1])
                dv01_2 = compute_dv01(val_date, t2, 1e6, mkt_rates_pct[t2])
                dv01_3 = compute_dv01(val_date, t3, 1e6, mkt_rates_pct[t3]) if is_fly else 0.0

                anc = anchor_w.value
                an = anchor_n_w.value
                target = target_dv01_w.value

                if not is_fly:
                    # Spread: two legs
                    if anc == "Leg1":
                        n1 = an
                        # n2 * dv01_2 = n1 * dv01_1 - target (target dv01 net = 0)
                        n2 = (n1 * dv01_1 - target) / dv01_2
                        print(f"Leg1 N: {n1:,.0f}  Leg2 N: {n2:,.0f}")
                    else:
                        n2 = an
                        n1 = (n2 * dv01_2 + target) / dv01_1
                        print(f"Leg1 N: {n1:,.0f}  Leg2 N: {n2:,.0f}")
                else:
                    # Fly: three legs
                    if anc == "Leg2":
                        n2 = an
                        # solve n1, n3 to net DV01 = target, keep n1=n3 for simple fly
                        # Typical fly: receive wings, pay body
                        n1 = n2 * dv01_2 / (dv01_1 + dv01_3)
                        n3 = n1
                        print(f"Leg1 N: {n1:,.0f}  Leg2 N: {n2:,.0f}  Leg3 N: {n3:,.0f}")
                    else:
                        print("For fly, anchor on Leg2 (body).")
            except Exception as e:
                print(f"Error: {e}")

    solve_btn.on_click(solve_hedge)

    leg3_box = widgets.VBox([leg3_w])
    def toggle_leg3(change):
        leg3_box.layout.display = "" if mode_w.value == "Fly" else "none"
    mode_w.observe(toggle_leg3, "value")
    toggle_leg3(None)

    hedge_box = widgets.VBox([
        hedge_title, mode_w,
        widgets.HBox([leg1_w, leg2_w, leg3_box]),
        widgets.HBox([anchor_w, anchor_n_w, target_dv01_w]),
        solve_btn, out_hedge
    ])

    return widgets.VBox([
        widgets.HTML("<b>DV01 per Tenor (at market rate)</b>"),
        widgets.HBox([notional_w, sign_w]),
        out_dv01,
        widgets.HTML("<hr>"),
        hedge_box
    ])

print("DV01/Hedge tab ready.")


In [ ]:
# Cell 9: Chart tab (plotly, 1Y lookback)

import os

def load_historical_data():
    """
    Load from xlsx if exists (Sheet1, cols: dates + tenor cols incl 18mo,2y, TNA).
    Falls back to 1Y synthetic data.
    """
    col_map = {
        "1m": 1, "2m": 2, "3m": 3, "4m": 4, "5m": 5, "6m": 6,
        "9m": 9, "10m": 10, "11m": 11, "12m": 12, "18m": 18, "18mo": 18,
        "2y": 24, "24m": 24
    }
    if os.path.exists(DATA_PATH):
        try:
            df = pd.read_excel(DATA_PATH, sheet_name="Sheet1", index_col=0, parse_dates=True)
            df.columns = [c.strip().lower() for c in df.columns]
            rename = {}
            for col in df.columns:
                if col in col_map:
                    rename[col] = col_map[col]
            df = df.rename(columns=rename)
            available = [c for c in TENORS if c in df.columns]
            return df[available]
        except Exception as e:
            print(f"Could not load {DATA_PATH}: {e}. Using synthetic data.")

    # Synthetic: 1Y of daily data ending today, random walk around OIS defaults
    dates = pd.date_range(end=date.today(), periods=252, freq="B")
    np.random.seed(42)
    data = {}
    for tm in TENORS:
        base = OIS_DEFAULTS[tm] / 100.0
        noise = np.random.randn(252).cumsum() * 0.0005
        noise -= noise[-1]  # end at base
        data[tm] = (base + noise) * 100.0  # back to pct
    return pd.DataFrame(data, index=dates)

def make_chart_tab(fv_df, mkt_rates_pct, weights):
    """Build interactive plotly chart widget."""
    hist_df = load_historical_data()

    tenor_dd = widgets.Dropdown(
        options=list(zip(TENOR_LABELS, TENORS)),
        description="Tenor:",
        value=3,
        style={"description_width": "60px"}
    )
    out_chart = widgets.Output()

    def update_chart(change=None):
        tm = tenor_dd.value
        tl = TENOR_LABELS[TENORS.index(tm)]

        # Historical mkt line
        if tm in hist_df.columns:
            hist_series = hist_df[tm].dropna()
        else:
            hist_series = pd.Series(dtype=float)

        # Weighted FV (scalar for selected tenor)
        path_names = list(fv_df.index)
        wt_arr = np.array([weights.get(p, 0.0) for p in path_names])
        wt_total = wt_arr.sum()
        wt_norm = wt_arr / wt_total if wt_total > 0 else np.ones(len(path_names)) / len(path_names)
        weighted_fv = float(wt_norm @ fv_df[tl].values) * 100.0  # pct
        mkt_rate = mkt_rates_pct[tm]

        fig = go.Figure()

        # Historical mkt
        if len(hist_series) > 0:
            fig.add_trace(go.Scatter(
                x=hist_series.index, y=hist_series.values,
                name="Mkt", line=dict(color="black", dash="dash", width=1.5),
            ))

        # FV horizontal line (orange solid)
        x_range = [hist_series.index[0] if len(hist_series) > 0 else date.today(),
                   date.today()]
        fig.add_trace(go.Scatter(
            x=x_range, y=[weighted_fv, weighted_fv],
            name="FV (wtd)", line=dict(color="orange", width=2),
        ))

        # Top 5 paths by weight (dotted lines as horizontal levels)
        top5 = sorted(weights.items(), key=lambda x: x[1], reverse=True)[:5]
        colors = ["#1f77b4", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
        for idx, (pname, wt) in enumerate(top5):
            if pname in fv_df.index:
                prate = float(fv_df.loc[pname, tl]) * 100.0
                fig.add_trace(go.Scatter(
                    x=x_range, y=[prate, prate],
                    name=f"{pname} ({wt:.1f}%)",
                    line=dict(color=colors[idx % 5], dash="dot", width=1),
                ))

        fig.update_layout(
            title=f"SOFR OIS {tl} — 1Y Lookback",
            xaxis_title="Date", yaxis_title="Rate (%)",
            template="plotly_white",
            legend=dict(orientation="h", y=-0.2),
            height=450,
        )

        with out_chart:
            out_chart.clear_output(wait=True)
            fig.show()

    tenor_dd.observe(update_chart, "value")
    update_chart()

    return widgets.VBox([tenor_dd, out_chart])

print("Chart tab ready.")


In [ ]:
# Cell 10: Main UI — Path table + inputs + output tabs

def build_ui():
    """Assemble full model UI."""
    # ---- Inputs section ----
    val_date_w = widgets.DatePicker(value=date.today(), description="Val Date:")
    effr_w = widgets.FloatText(value=EFFR_DEFAULT * 100, description="EFFR (%):", step=0.01,
                               style={"description_width": "80px"}, layout=widgets.Layout(width="180px"))
    wirp_increment_w = widgets.IntText(value=25, description="WIRP incr(bp):",
                                       style={"description_width": "100px"}, layout=widgets.Layout(width="180px"))

    # OIS mid inputs (one per tenor)
    ois_inputs = {}
    ois_boxes = []
    for tm, tl in zip(TENORS, TENOR_LABELS):
        w = widgets.FloatText(value=OIS_DEFAULTS[tm], description=f"{tl}:",
                              step=0.001, style={"description_width": "35px"},
                              layout=widgets.Layout(width="130px"))
        ois_inputs[tm] = w
        ois_boxes.append(w)

    ois_section = widgets.VBox([
        widgets.HTML("<b>OIS Mids (% TNA):</b>"),
        widgets.HBox(ois_boxes[:6]),
        widgets.HBox(ois_boxes[6:]),
    ])

    # ---- Path table ----
    # Each path: name field, weight field, 13 meeting bp fields
    path_rows_state = []  # list of dicts with widgets

    def make_path_row(name, bps, weight):
        name_w = widgets.Text(value=name, layout=widgets.Layout(width="140px"))
        weight_w = widgets.FloatText(value=weight, layout=widgets.Layout(width="65px"))
        bp_ws = [widgets.IntText(value=int(bps[i]), layout=widgets.Layout(width="55px"))
                 for i in range(N_MEETINGS)]
        del_btn = widgets.Button(description="X", button_style="danger",
                                 layout=widgets.Layout(width="35px", height="28px"))
        row = {"name": name_w, "weight": weight_w, "bps": bp_ws, "del": del_btn}
        return row

    def build_path_table_widget():
        header = widgets.HBox(
            [widgets.HTML("<b>Path Name</b>", layout=widgets.Layout(width="140px")),
             widgets.HTML("<b>Wt%</b>", layout=widgets.Layout(width="65px"))] +
            [widgets.HTML(f"<b>{lbl}</b>", layout=widgets.Layout(width="55px")) for lbl in MTG_LABELS] +
            [widgets.HTML("", layout=widgets.Layout(width="35px"))]
        )
        rows_widgets = [header]
        for r in path_rows_state:
            row_box = widgets.HBox([r["name"], r["weight"]] + r["bps"] + [r["del"]])
            rows_widgets.append(row_box)
        return widgets.VBox(rows_widgets)

    path_table_out = widgets.Output()

    def refresh_path_table():
        with path_table_out:
            path_table_out.clear_output(wait=True)
            display(build_path_table_widget())

    # Init rows
    default_wt = 100.0 / len(DEFAULT_PATHS)
    for name, bps in DEFAULT_PATHS.items():
        row = make_path_row(name, bps, default_wt)
        path_rows_state.append(row)
        def make_del_handler(r):
            def handler(_):
                path_rows_state.remove(r)
                refresh_path_table()
            return handler
        row["del"].on_click(make_del_handler(row))

    add_path_btn = widgets.Button(description="+ Add Path", button_style="success")
    def on_add_path(_):
        row = make_path_row("New Path", _zeros(), 0.0)
        path_rows_state.append(row)
        def make_del_handler(r):
            def handler(_):
                path_rows_state.remove(r)
                refresh_path_table()
            return handler
        row["del"].on_click(make_del_handler(row))
        refresh_path_table()
    add_path_btn.on_click(on_add_path)

    refresh_path_table()

    # ---- Output area ----
    out_fv = widgets.Output()
    out_wirp = widgets.Output()
    out_spread = widgets.Output()
    out_fly = widgets.Output()

    # RV view dropdown (shared for spread+fly)
    rv_view_dd = widgets.Dropdown(
        options=["FV"] + list(DEFAULT_PATHS.keys()),
        value="FV", description="View:",
        style={"description_width": "50px"},
        layout=widgets.Layout(width="200px")
    )

    compute_btn = widgets.Button(description="Compute FV", button_style="primary",
                                 layout=widgets.Layout(width="140px"))

    # State holder for last computed FV
    _state = {"fv_df": None, "weights": {}}

    def on_compute(_):
        # Gather state from widgets
        vd = val_date_w.value
        if isinstance(vd, str):
            from datetime import datetime
            vd = datetime.strptime(vd, "%Y-%m-%d").date()
        effr = effr_w.value / 100.0
        ois_mids = {tm: ois_inputs[tm].value for tm in TENORS}
        wirp_inc = wirp_increment_w.value

        # Build paths df from table
        rows = {}
        weights = {}
        for r in path_rows_state:
            pname = r["name"].value.strip() or "Path"
            bps = [r["bps"][i].value for i in range(N_MEETINGS)]
            wt = r["weight"].value
            rows[pname] = bps
            weights[pname] = wt
        if not rows:
            return
        paths_df = pd.DataFrame(rows, index=MTG_LABELS).T

        # Compute FVs
        fv_df = compute_all_fvs(vd, paths_df, effr)
        _state["fv_df"] = fv_df
        _state["weights"] = weights

        # Update RV dropdown options
        rv_view_dd.options = ["FV"] + list(rows.keys())

        # Render FV table
        with out_fv:
            out_fv.clear_output(wait=True)
            display(HTML(render_fv_table(fv_df, ois_mids, weights)))

        # Render WIRP
        wirp_df = wirp_bootstrap(vd, ois_mids, effr, wirp_inc)
        with out_wirp:
            out_wirp.clear_output(wait=True)
            display(wirp_df.to_string(index=False))

        # Render spread/fly with current view
        _refresh_rv(fv_df, ois_mids, weights, rv_view_dd.value)

    def _refresh_rv(fv_df, ois_mids, weights, view):
        with out_spread:
            out_spread.clear_output(wait=True)
            display(HTML(render_spread_rv(fv_df, ois_mids, weights, view)))
        with out_fly:
            out_fly.clear_output(wait=True)
            display(HTML(render_fly_rv(fv_df, ois_mids, weights, view)))

    def on_rv_view_change(change):
        if _state["fv_df"] is not None:
            ois_mids = {tm: ois_inputs[tm].value for tm in TENORS}
            _refresh_rv(_state["fv_df"], ois_mids, _state["weights"], change["new"])

    rv_view_dd.observe(on_rv_view_change, "value")
    compute_btn.on_click(on_compute)

    # ---- Assemble tabs ----
    tab_fv = widgets.VBox([out_fv])
    tab_wirp = widgets.VBox([out_wirp])
    tab_spread = widgets.VBox([widgets.HBox([widgets.HTML("<b>Spread RV — View:</b>"), rv_view_dd]), out_spread])
    tab_fly_rv = widgets.VBox([out_fly])

    # DV01 tab — lazy built on compute
    dv01_out = widgets.Output()

    def build_dv01_tab():
        ois_mids = {tm: ois_inputs[tm].value for tm in TENORS}
        vd = val_date_w.value
        if isinstance(vd, str):
            from datetime import datetime
            vd = datetime.strptime(vd, "%Y-%m-%d").date()
        with dv01_out:
            dv01_out.clear_output(wait=True)
            display(make_dv01_tab(vd, ois_mids))

    compute_btn.on_click(lambda _: build_dv01_tab())

    tab_dv01 = widgets.VBox([dv01_out])

    # Chart tab
    chart_out = widgets.Output()
    def build_chart():
        if _state["fv_df"] is not None:
            ois_mids = {tm: ois_inputs[tm].value for tm in TENORS}
            with chart_out:
                chart_out.clear_output(wait=True)
                display(make_chart_tab(_state["fv_df"], ois_mids, _state["weights"]))
    compute_btn.on_click(lambda _: build_chart())

    tab_chart = widgets.VBox([chart_out])

    tabs = widgets.Tab(children=[tab_fv, tab_wirp, tab_spread, tab_fly_rv, tab_dv01, tab_chart])
    for i, title in enumerate(["FV & Deltas", "WIRP", "Spread RV", "Fly RV", "DV01 & Hedge", "Chart"]):
        tabs.set_title(i, title)

    ui = widgets.VBox([
        widgets.HTML("<h2>SOFR OIS FV Model</h2>"),
        widgets.HBox([val_date_w, effr_w, wirp_increment_w]),
        ois_section,
        widgets.HTML("<hr><b>Paths:</b>"),
        path_table_out,
        widgets.HBox([add_path_btn, compute_btn]),
        widgets.HTML("<hr>"),
        tabs,
    ])

    return ui

ui = build_ui()
display(ui)
